# STURM-Flood Inference & Visualization Notebook

This notebook performs flood extent inference on Sentinel-1 and Sentinel-2 tiles using pretrained U-Net models, and computes metrics when ground-truth is available.

Models and datasets are available on Zenodo:
- Sentinel-1: [DOI:10.5281/zenodo.15189664](https://doi.org/10.5281/zenodo.15189664)
- Sentinel-2: [DOI:10.5281/zenodo.15189633](https://doi.org/10.5281/zenodo.15189633)
- Dataset: [DOI:10.5281/zenodo.12748982](https://doi.org/10.5281/zenodo.12748982)

## Attribution & License

This notebook is **adapted from the inference notebook of the STURM-Flood project**:

- Source: STURM-WEO/STURM-Flood, https://github.com/STURM-WEO/STURM-Flood
- Paper: Notarangelo, N. M., Wirion, C., & van Winsen, F. (2025). *STURM-Flood: A curated dataset for deep learning-based flood extent mapping leveraging Sentinel-1 and Sentinel-2 imagery.* Big Earth Data. https://doi.org/10.1080/20964471.2025.2458714

The U-Net architecture and the core inference, I/O and visualization functions are **imported from the STURM-Flood repository (cloned at runtime)** and are not redistributed here. Pretrained model weights are downloaded from Zenodo.

**Changes made in this adaptation:**
- Added Google Drive integration for input tiles and output storage
- Added study-area configuration for the EMSR427 test sites (Karlstad, Göteborg, Kristianstad)
- Replaced the evaluation-metrics cell to report micro-averaged metrics matching the STURM-Flood paper (Table 7)
- Added renaming of prediction GeoTIFFs for compatibility with the downstream TileExplorer notebook

**License:** The original STURM-Flood repository is licensed under **CC BY-SA 4.0**. Because this notebook is an adaptation of that work, it is distributed under the **same CC BY-SA 4.0 license**. See https://creativecommons.org/licenses/by-sa/4.0/. (The original notebooks in this repository, authored by Stefan Edvinsson, are licensed under MIT as stated in the repository LICENSE.)


# Setup environment

Clone the STURM-Flood repository and install dependencies.
**Run this once** at the start of a new Colab session.

In [ ]:
# Clone the STURM-Flood GitHub repo into /content/STURM-Flood
# This gives us: arch/ (U-Net model code), utils/ (helper functions)
!git clone https://github.com/STURM-WEO/STURM-Flood.git
%cd STURM-Flood

# Install tensorflow-io (needed for some data loading functions)
# --quiet hides the long installation log
!pip install tensorflow-io --quiet

# Configuration

**Edit this cell before running the rest of the notebook.**

- **Sensor toggles**: Choose S1, S2, or both
- **Data source**: `"upload"` (your own tiles) or `"zenodo"` (download the full dataset)
- **Ground truth**: Whether you have flood masks to compare against

In [ ]:
import os
import shutil

# ============================================================
# CONFIGURATION - Edit these settings before running!
# ============================================================

# --- Sensor selection ---
# Sentinel-1 = SAR radar data (2 bands: VV + VH, works through clouds)
# Sentinel-2 = Optical/multispectral data (9 bands, needs clear sky)
use_S1 = True    # Enable Sentinel-1 inference
use_S2 = False   # Enable Sentinel-2 inference

# --- Data source ---
# "drive"   = Copy tiles from Google Drive folder (recommended when running in Colab)
# "upload"  = Upload your own .tif tiles via a file dialog
#             The notebook creates folders and sorts files automatically.
# "zenodo"  = Download the full STURM-Flood dataset from Zenodo (~large download)
data_source = "drive"   # "drive", "upload" or "zenodo"

# --- Ground truth ---
# True  = You have flood mask files to compare against → gives accuracy metrics
# False = No masks, just generate flood predictions
with_gt = True

# ---Study area: 'karlstad', 'goteborg' or 'kristianstad'. Change before each run
area = 'goteborg_final'

# --- Number of test tiles (only used with data_source = "zenodo") ---
# How many random tiles to sample. Set to None for ALL tiles.
n_sample = 10

# ============================================================
print("=" * 50)
print("CONFIGURATION SUMMARY")
print("=" * 50)
print(f"  Sentinel-1 (SAR):      {'ON' if use_S1 else 'OFF'}")
print(f"  Sentinel-2 (Optical):  {'ON' if use_S2 else 'OFF'}")
print(f"  Data source:           {data_source}")
print(f"  Ground truth masks:    {'YES' if with_gt else 'NO'}")
if data_source == "zenodo":
    print(f"  Tiles to sample:       {n_sample if n_sample else 'ALL'}")
print("=" * 50)

if not use_S1 and not use_S2:
    raise ValueError("Enable at least one sensor! Set use_S1 or use_S2 to True.")

## Output folders on Google Drive

Create folder structure for predictions and metrics, mirroring Otsu/RF results.

In [ ]:
# ============================================================
# OUTPUT FOLDER STRUCTURE ON GOOGLE DRIVE
# Mirrors RF/Otsu results for easy cross-method comparison.
# ============================================================
from google.colab import drive as _drive
_drive.mount('/content/drive')

unet_output_base = f'/content/drive/MyDrive/Examensarbete/Unet_results/{area}'
drive_dirs = {
    'tables':  unet_output_base + '/tables',
    'geotiff': unet_output_base + '/geotiff_predictions',
}
import os as _os
for d in drive_dirs.values():
    _os.makedirs(d, exist_ok=True)
print('Drive output folders ready:')
for k, v in drive_dirs.items():
    print(f'  {k}: {v}')


## Upload your data

**Only needed if `data_source = "upload" or "drive".**

For `data_source = "drive"`: set `drive_dataset_path` in the cell below and run it. Tiles are copied automatically from Google Drive.

In [ ]:
# --- Upload tiles via file dialog OR Google Drive ---
# Two options:
#   data_source = "upload"  → opens file picker (slow for many files)
#   data_source = "drive"   → mounts Google Drive and copies from a folder (fast)
#
# For large datasets (1000+ tiles), use "drive" to avoid browser upload limits.

if data_source == "drive":
    # --- Mount Google Drive and copy dataset ---
    # Set drive_dataset_path to your zip file or folder in Drive.
    # Example: "/content/drive/MyDrive/EMSR427_dataset.zip"
    from google.colab import drive
    drive.mount('/content/drive')

    drive_dataset_path = "/content/drive/MyDrive/Examensarbete/Dataset_goteborg_final"  # <-- EDIT THIS PATH

    if drive_dataset_path.endswith('.zip'):
        # Unzip directly into /content/Dataset
        import zipfile
        print(f"Extracting {drive_dataset_path} ...")
        with zipfile.ZipFile(drive_dataset_path, 'r') as z:
            z.extractall('/content/Dataset')
        print(f"✓ Extracted to /content/Dataset")
    else:
        # Copy folder from Drive
        print(f"Copying from {drive_dataset_path} ...")
        shutil.copytree(drive_dataset_path, '/content/Dataset', dirs_exist_ok=True)
        print(f"✓ Copied to /content/Dataset")

    # Verify what was loaded
    for root, dirs, files_list in os.walk('/content/Dataset'):
        tifs = [f for f in files_list if f.endswith('.tif')]
        if tifs:
            print(f"  {root}: {len(tifs)} .tif files")

elif data_source == "upload":
    from google.colab import files

    def upload_to_folder(target_folder, description):
        """
        Opens a file dialog, uploads selected files, and moves them
        to the target folder. Returns the number of files uploaded.

        How it works:
        1. files.upload() opens a browser dialog where you pick files
        2. The files land in the current directory as a dict {filename: bytes}
        3. We move each file to the target folder with shutil.move()
        """
        os.makedirs(target_folder, exist_ok=True)
        print(f"\n{'='*50}")
        print(f"  >> {description}")
        print(f"  >> Select your .tif files in the dialog below")
        print(f"  >> Target folder: {target_folder}")
        print(f"{'='*50}")

        # This line opens the file picker dialog in your browser
        uploaded = files.upload()

        # Move each uploaded file to the target folder
        count = 0
        for filename in uploaded.keys():
            dest = os.path.join(target_folder, filename)
            shutil.move(filename, dest)
            count += 1
            print(f"  Moved: {filename} → {target_folder}/")

        print(f"  Total: {count} files uploaded to {target_folder}")
        return count

    # --- Step-by-step upload for each sensor and data type ---

    if use_S1:
        # Step 1: Upload S1 satellite tiles
        n = upload_to_folder(
            '/content/Dataset/Sentinel1/S1',
            'STEP 1: Upload Sentinel-1 SATELLITE TILES'
        )

        # Step 2: Upload S1 flood masks (only if ground truth is available)
        if with_gt:
            n = upload_to_folder(
                '/content/Dataset/Sentinel1/Floodmaps',
                'STEP 2: Upload Sentinel-1 FLOOD MASKS (ground truth)'
            )

    if use_S2:
        # Step 3: Upload S2 satellite tiles
        n = upload_to_folder(
            '/content/Dataset/Sentinel2/S2',
            'STEP 3: Upload Sentinel-2 SATELLITE TILES'
        )

        # Step 4: Upload S2 flood masks (only if ground truth is available)
        if with_gt:
            n = upload_to_folder(
                '/content/Dataset/Sentinel2/Floodmaps',
                'STEP 4: Upload Sentinel-2 FLOOD MASKS (ground truth)'
            )

    print("\n" + "=" * 50)
    print("UPLOAD COMPLETE!")
    print("=" * 50)

else:
    print("Skipping upload (data_source = 'zenodo').")

# Import dependencies

In [ ]:
# Standard Python libraries:
# - os, sys: file paths and system operations
# - glob: find files matching a pattern (e.g. all .tif files)
# - tqdm: progress bar for loops
# - numpy: numerical arrays (images are numpy arrays)
# - pandas: data tables for metadata and results
# - requests: downloading files from internet
import os
import sys
import glob

from tqdm import tqdm
import numpy as np
import pandas as pd
import requests

# Functions

In [ ]:
# --- Add the U-Net model architecture to Python's search path ---
# The 'arch' folder (from the cloned repo) contains model.py with the U-Net definition.
# sys.path.append tells Python: "also look in this folder when importing"
arch_dir = './arch'
sys.path.append(arch_dir)
from model import unet_model

def rebuild_model(params):
    """
    Rebuild a U-Net model and load pretrained weights.

    1. Creates a fresh U-Net with the correct architecture
    2. Loads pretrained weights (.hdf5) - the model's "learned knowledge"

    Parameters:
    -----------
    params : dict
        'input_shape'  : (128, 128, bands) - S1=2 bands (VV,VH), S2=9 bands
        'n_blocks'     : encoder/decoder depth (S1=6, S2=5)
        'weights_path' : path to .hdf5 weights file

    Returns:
    --------
    model : compiled U-Net ready for inference
    """
    model = unet_model(
        n_classes=2,                          # Binary: flood vs no-flood
        tile_width=params['input_shape'][0],   # 128 pixels
        tile_height=params['input_shape'][1],  # 128 pixels
        n_bands=params['input_shape'][2],      # 2 for S1, 9 for S2
        n_blocks=params['n_blocks'],           # Network depth
        class_weight_list=[1, 1],              # Equal weight for both classes
        normalize_inputs=False                 # Data already normalized
    )
    model.load_weights(params['weights_path'])
    return model

In [ ]:
# --- Import utility functions from utils/ (from the cloned repo) ---
# - download_from_zenodo: downloads files with progress bar
# - extract_tarfile/zipfile: unpacks archives
# - run_inference: feeds one tile through model → flood prediction + metrics
# - save_visualizations: creates comparison images (satellite | prediction | mask)
sys.path.append('./utils')
from utils.io import download_from_zenodo, extract_tarfile, extract_zipfile
from utils.inference_metrics import run_inference
from utils.visualization import save_visualizations

# STURM-Flood dataset download (Zenodo)

**Only runs if `data_source = "zenodo"`.** Skip if you uploaded your own data.

In [ ]:
# --- Download STURM-Flood dataset from Zenodo ---
if data_source == "zenodo":
    dataset_url = "https://zenodo.org/records/12748983/files/Dataset.zip"
    dataset_dir = './STURM-Flood'
    os.makedirs(dataset_dir, exist_ok=True)
    dataset_zip_path = os.path.join(dataset_dir, "STURM-Flood.zip")

    print("Downloading STURM-Flood dataset...")
    download_from_zenodo(dataset_url, dataset_zip_path)
    print("Extracting...")
    extract_zipfile(dataset_zip_path, dataset_dir)
    print("Dataset ready!")
else:
    print("Skipping (data_source = 'upload')")

# U-Net model download & setup

Downloads pretrained weights from Zenodo and builds the U-Net models.
Only downloads models for sensors you enabled.

In [ ]:
# --- Download pretrained U-Net model weights ---
# S1 and S2 have separate models trained on different data types.
s1_model_url = "https://zenodo.org/records/15189665/files/S1_model.tar.gz"
s2_model_url = "https://zenodo.org/records/15189634/files/S2_model.tar.gz"

s1_model_dir = './unet/sentinel1_model'
s2_model_dir = './unet/sentinel2_model'

if use_S1:
    os.makedirs(s1_model_dir, exist_ok=True)
    os.makedirs(os.path.join(s1_model_dir, 'unet/1'), exist_ok=True)
    s1_model_tar = os.path.join(s1_model_dir, 'model.tar.gz')
    print("Downloading Sentinel-1 model...")
    download_from_zenodo(s1_model_url, s1_model_tar)
    extract_tarfile(s1_model_tar, s1_model_dir)
    print("S1 model ready!")
else:
    print("Skipping S1 model (use_S1 = False)")

if use_S2:
    os.makedirs(s2_model_dir, exist_ok=True)
    os.makedirs(os.path.join(s2_model_dir, 'unet/1'), exist_ok=True)
    s2_model_tar = os.path.join(s2_model_dir, 'model.tar.gz')
    print("Downloading Sentinel-2 model...")
    download_from_zenodo(s2_model_url, s2_model_tar)
    extract_tarfile(s2_model_tar, s2_model_dir)
    print("S2 model ready!")
else:
    print("Skipping S2 model (use_S2 = False)")

In [ ]:
# --- Model parameters (must match original training) ---
model_params = {
    'Sentinel-1': {
        'input_shape': (128, 128, 2),   # 2 bands: VV, VH
        'n_blocks': 6,
        'weights_path': os.path.join(s1_model_dir, 'unet/1/model_weights.hdf5')
    },
    'Sentinel-2': {
        'input_shape': (128, 128, 9),   # 9 spectral bands
        'n_blocks': 5,
        'weights_path': os.path.join(s2_model_dir, 'unet/1/model_weights.hdf5')
    }
}

# --- Build models for enabled sensors ---
s1_model = None
s2_model = None

if use_S1:
    print("Building Sentinel-1 U-Net...")
    s1_model = rebuild_model(model_params['Sentinel-1'])
    print("  S1 model loaded!")

if use_S2:
    print("Building Sentinel-2 U-Net...")
    s2_model = rebuild_model(model_params['Sentinel-2'])
    print("  S2 model loaded!")

# Inference

## Load tile lists

Scans the data folders to find all tiles. Uses `glob` for uploaded data or metadata CSVs for the Zenodo dataset.

In [ ]:
# Make sure we're in the right directory
%cd /content/STURM-Flood

# --- Output directories ---
output_dir = drive_dirs['geotiff']  # Save GeoTIFF predictions directly to Drive
visualization_dir = './tile_visualizations'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(visualization_dir, exist_ok=True)

s1_test_df = None
s2_test_df = None
composite_dirs = {}
mask_dirs = {}

if data_source == "upload":
    # --- UPLOAD MODE: find tiles with glob ---
    # glob.glob("*.tif") returns a list of all .tif files in the folder.
    # We extract just the filename with os.path.basename() and put it in a DataFrame.

    if use_S1:
        composite_dirs['Sentinel-1'] = "/content/Dataset/Sentinel1/S1"
        mask_dirs['Sentinel-1'] = "/content/Dataset/Sentinel1/Floodmaps"
        s1_tiles = glob.glob("/content/Dataset/Sentinel1/S1/*.tif")
        s1_test_df = pd.DataFrame({'tile_id': [os.path.basename(f) for f in s1_tiles]})
        print(f"Sentinel-1: {len(s1_test_df)} tiles found")
        if len(s1_test_df) == 0:
            print("  WARNING: No .tif files found! Did the upload work?")

    if use_S2:
        composite_dirs['Sentinel-2'] = "/content/Dataset/Sentinel2/S2"
        mask_dirs['Sentinel-2'] = "/content/Dataset/Sentinel2/Floodmaps"
        s2_tiles = glob.glob("/content/Dataset/Sentinel2/S2/*.tif")
        s2_test_df = pd.DataFrame({'tile_id': [os.path.basename(f) for f in s2_tiles]})
        print(f"Sentinel-2: {len(s2_test_df)} tiles found")
        if len(s2_test_df) == 0:
            print("  WARNING: No .tif files found! Did the upload work?")

elif data_source == "zenodo":
    # --- ZENODO MODE: read metadata CSVs ---
    composite_dirs = {
        'Sentinel-1': "./STURM-Flood/Dataset/Sentinel1/S1",
        'Sentinel-2': "./STURM-Flood/Dataset/Sentinel2/S2"
    }
    mask_dirs = {
        'Sentinel-1': "./STURM-Flood/Dataset/Sentinel1/Floodmaps",
        'Sentinel-2': "./STURM-Flood/Dataset/Sentinel2/Floodmaps"
    }

    if use_S1:
        s1_metadata = pd.read_csv("./STURM-Flood/Dataset/Sentinel1_metadata.csv")
        s1_test_df = s1_metadata.sample(n=n_sample, random_state=42).reset_index(drop=True) if n_sample else s1_metadata
        print(f"Sentinel-1: {len(s1_test_df)} tiles from CSV")

    if use_S2:
        s2_metadata = pd.read_csv("./STURM-Flood/Dataset/Sentinel2_metadata.csv")
        s2_test_df = s2_metadata.sample(n=n_sample, random_state=42).reset_index(drop=True) if n_sample else s2_metadata
        print(f"Sentinel-2: {len(s2_test_df)} tiles from CSV")

elif data_source == "drive":
    if use_S1:
        composite_dirs['Sentinel-1'] = "/content/Dataset/Sentinel1/S1"
        mask_dirs['Sentinel-1'] = "/content/Dataset/Sentinel1/Floodmaps"
        s1_tiles = glob.glob("/content/Dataset/Sentinel1/S1/*.tif")
        s1_test_df = pd.DataFrame({'tile_id': [os.path.basename(f) for f in s1_tiles]})
        print(f"Sentinel-1: {len(s1_test_df)} tiles found")
        if len(s1_test_df) == 0:
            print("  WARNING: No .tif files found! Check drive_dataset_path.")

## Run inference

For each tile:
1. Load 128×128 satellite image → feed through U-Net → probability map (0.0–1.0)
2. Pixels > `score_threshold` (0.5) = **flood**
3. If ground truth exists → calculate IoU, F1, Precision, Recall

In [ ]:
# --- Main inference loop ---
score_threshold = 0.5  # Flood probability cutoff (0.5 = 50%)
results = []

# Build list of sensor jobs (only enabled sensors with tiles)
datasets_to_run = []
if use_S1 and s1_test_df is not None and len(s1_test_df) > 0:
    datasets_to_run.append(('Sentinel-1', s1_test_df, s1_model))
if use_S2 and s2_test_df is not None and len(s2_test_df) > 0:
    datasets_to_run.append(('Sentinel-2', s2_test_df, s2_model))

if not datasets_to_run:
    print("ERROR: No tiles to process! Check config and uploaded files.")

for dataset, df, model in datasets_to_run:
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {dataset}"):
        tile = row['tile_id']
        metrics = run_inference(
            tile, dataset, model,
            composite_dirs, mask_dirs, output_dir,
            score_threshold, with_gt
        )
        if metrics:
            metrics['tile_id'] = tile
            metrics['Dataset'] = dataset
            results.append(metrics)

# --- Save results ---
if with_gt and results:
    results_df = pd.DataFrame(results)
    results_df.to_csv('inference_metrics.csv', index=False)
    print(f"\nDone! {len(results)} tiles processed. Metrics → inference_metrics.csv")
elif not with_gt:
    print(f"\nDone! Predictions saved (no metrics, with_gt = False)")
else:
    print("\nWARNING: No results. Check data paths and filenames.")

## Metrics summary

Per-tile and overall accuracy. Only works if `with_gt = True`.

- **Precision**: Of predicted flood pixels, how many were correct?
- **Recall**: Of actual flood pixels, how many did the model find?
- **F1**: Balanced score (harmonic mean of Precision & Recall)
- **IoU**: Overlap between prediction and ground truth

In [ ]:
# @title
print(results_df)

print(results_df[['F1', 'Precision', 'Recall', 'IoU']].mean())

# Sum TP, TN, FP, FN across all tiles
total_TP = results_df['TP'].sum()
total_TN = results_df['TN'].sum()
total_FP = results_df['FP'].sum()
total_FN = results_df['FN'].sum()

# Compute overall (micro-averaged) metrics
precision = total_TP / (total_TP + total_FP)
recall = total_TP / (total_TP + total_FN)
f1 = 2 * (precision * recall) / (precision + recall)
accuracy = (total_TP + total_TN) / (total_TP + total_TN + total_FP + total_FN)

print(f"Overall Precision: {precision:.3f}")
print(f"Overall Recall: {recall:.3f}")
print(f"Overall F1: {f1:.3f}")
print(f"Overall Accuracy: {accuracy:.3f}")

In [ ]:
# @title
# =============================================================================
# EVALUATION METRICS: matching STURM-Flood paper Table 7 format
# =============================================================================
# Requires: results_df with columns TP, TN, FP, FN, tile_id, Dataset
# =============================================================================

# --- PER-TILE METRICS ---
print("=" * 70)
print("PER-TILE METRICS")
print("=" * 70)
print(results_df[['F1', 'Precision', 'Recall', 'IoU', 'TP', 'TN', 'FP', 'FN', 'tile_id', 'Dataset']])

# --- AVERAGE PER SENSOR ---
print("\n" + "=" * 70)
print("AVERAGE PER SENSOR")
print("=" * 70)
print(results_df.groupby('Dataset')[['F1', 'Precision', 'Recall', 'IoU']].mean())

# --- MICRO-AVERAGED OVERALL ---
total_TP = results_df['TP'].sum()
total_TN = results_df['TN'].sum()
total_FP = results_df['FP'].sum()
total_FN = results_df['FN'].sum()

micro_precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0
micro_recall = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0
micro_f1 = 2 * (micro_precision * micro_recall) / (micro_precision + micro_recall) if (micro_precision + micro_recall) > 0 else 0
micro_iou = total_TP / (total_TP + total_FP + total_FN) if (total_TP + total_FP + total_FN) > 0 else 0
accuracy = (total_TP + total_TN) / (total_TP + total_TN + total_FP + total_FN)

print("\n" + "=" * 70)
print("OVERALL METRICS (micro-averaged)")
print("=" * 70)
print(f"  Precision: {micro_precision:.3f}")
print(f"  Recall:    {micro_recall:.3f}")
print(f"  F1 Score:  {micro_f1:.3f}")
print(f"  IoU:       {micro_iou:.3f}")
print(f"  Accuracy:  {accuracy:.3f}")
print(f"\n  TP={total_TP:,}  TN={total_TN:,}  FP={total_FP:,}  FN={total_FN:,}")

# =============================================================================
# PER-CLASS METRICS: matches STURM-Flood paper Table 7
# =============================================================================
# The paper reports Precision, Recall, and F1 for EACH class separately,
# then Macro avg (unweighted mean of both classes) and Weighted avg
# (weighted by class support, i.e. number of actual pixels per class).
#
# "Water" = positive class (TP/FP/FN as usual)
# "Non-Water" = negative class (treat TN as TP for non-water, etc.)
# =============================================================================

# Water class metrics (positive class)
water_precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0
water_recall    = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0
water_f1 = 2 * (water_precision * water_recall) / (water_precision + water_recall) if (water_precision + water_recall) > 0 else 0
water_iou = total_TP / (total_TP + total_FP + total_FN) if (total_TP + total_FP + total_FN) > 0 else 0

# Non-Water class metrics (treat TN as the "true positives" for non-water)
nw_precision = total_TN / (total_TN + total_FN) if (total_TN + total_FN) > 0 else 0
nw_recall    = total_TN / (total_TN + total_FP) if (total_TN + total_FP) > 0 else 0
nw_f1 = 2 * (nw_precision * nw_recall) / (nw_precision + nw_recall) if (nw_precision + nw_recall) > 0 else 0
nw_iou = total_TN / (total_TN + total_FN + total_FP) if (total_TN + total_FN + total_FP) > 0 else 0

# Macro average: unweighted mean of both classes
macro_precision = (water_precision + nw_precision) / 2
macro_recall    = (water_recall + nw_recall) / 2
macro_f1        = (water_f1 + nw_f1) / 2
macro_iou       = (water_iou + nw_iou) / 2

# Weighted average: weighted by support (number of actual pixels per class)
water_support = total_TP + total_FN       # all actual water pixels
nw_support    = total_TN + total_FP       # all actual non-water pixels
total_support = water_support + nw_support

weighted_precision = (water_precision * water_support + nw_precision * nw_support) / total_support
weighted_recall    = (water_recall * water_support + nw_recall * nw_support) / total_support
weighted_f1        = (water_f1 * water_support + nw_f1 * nw_support) / total_support
weighted_iou       = (water_iou * water_support + nw_iou * nw_support) / total_support

# --- Print in same format as paper Table 7 ---
print("\n" + "=" * 70)
print("PER-CLASS METRICS (paper Table 7 format)")
print("=" * 70)
print(f"{'Metric':<12} {'Non-Water':>12} {'Water':>12} {'Macro avg':>12} {'Weighted avg':>14}")
print("-" * 64)
print(f"{'Precision':<12} {nw_precision:>11.2%} {water_precision:>11.2%} {macro_precision:>11.2%} {weighted_precision:>13.2%}")
print(f"{'Recall':<12} {nw_recall:>11.2%} {water_recall:>11.2%} {macro_recall:>11.2%} {weighted_recall:>13.2%}")
print(f"{'F1-Score':<12} {nw_f1:>12.4f} {water_f1:>12.4f} {macro_f1:>12.4f} {weighted_f1:>14.4f}")
print(f"{'IoU':<12} {nw_iou:>12.4f} {water_iou:>12.4f} {macro_iou:>12.4f} {weighted_iou:>14.4f}")

# --- Comparison with paper's reported values ---
print("\n" + "=" * 70)
print("COMPARISON WITH STURM-FLOOD PAPER (Table 7, Sentinel-1)")
print("=" * 70)
print(f"{'Metric':<12} {'Paper':>12} {'EMSR427':>12} {'Diff':>12}")
print("-" * 50)

paper = {
    'Water Prec':    (0.7465, water_precision),
    'Water Rec':     (0.6553, water_recall),
    'Water F1':      (0.6979, water_f1),
    'NW Prec':       (0.8666, nw_precision),
    'NW Rec':        (0.9096, nw_recall),
    'NW F1':         (0.8875, nw_f1),
    'Macro F1':      (0.7927, macro_f1),
    'Accuracy':      (0.8361, accuracy),
}

for name, (paper_val, our_val) in paper.items():
    diff = our_val - paper_val
    sign = "+" if diff >= 0 else ""
    print(f"{name:<12} {paper_val:>11.4f} {our_val:>11.4f} {sign}{diff:>10.4f}")

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)
print(f"                    Predicted")
print(f"{'Actual':<12} {'Non-Water':>12} {'Water':>12}")
print(f"{'Non-Water':<12} {total_TN:>12,} {total_FP:>12,}")
print(f"{'Water':<12} {total_FN:>12,} {total_TP:>12,}")

## Visualize results

Side-by-side images: `[Satellite] | [Prediction] | [Ground Truth]`
Saved to `tile_visualizations/` and shown inline.

In [ ]:
# --- Visualizations ---
if use_S1:
    if with_gt and 'results_df' in dir() and len(results_df[results_df['Dataset'] == 'Sentinel-1']) > 0:
        save_visualizations(
            results_df[results_df['Dataset'] == 'Sentinel-1'],
            'Sentinel-1', visualization_dir, composite_dirs,
            output_dir, mask_dirs, is_s2=False, with_gt=True, show_inline=True
        )
    elif not with_gt and s1_test_df is not None and len(s1_test_df) > 0:
        save_visualizations(
            s1_test_df, 'Sentinel-1', visualization_dir, composite_dirs,
            output_dir, mask_dirs, is_s2=False, with_gt=False, show_inline=True
        )
    print("S1 visualizations saved!")

if use_S2:
    if with_gt and 'results_df' in dir() and len(results_df[results_df['Dataset'] == 'Sentinel-2']) > 0:
        save_visualizations(
            results_df[results_df['Dataset'] == 'Sentinel-2'],
            'Sentinel-2', visualization_dir, composite_dirs,
            output_dir, mask_dirs, is_s2=True, with_gt=True, show_inline=True
        )
    elif not with_gt and s2_test_df is not None and len(s2_test_df) > 0:
        save_visualizations(
            s2_test_df, 'Sentinel-2', visualization_dir, composite_dirs,
            output_dir, mask_dirs, is_s2=True, with_gt=False, show_inline=True
        )
    print("S2 visualizations saved!")

print(f"\nSaved to: {visualization_dir}")

## Rename prediction files

Rename binary GeoTIFF predictions to match input tile filenames for TileExplorer compatibility.

In [ ]:
import os, glob, shutil

drive_geotiff = drive_dirs['geotiff']

renamed = 0

# Rename binary files: sentinel-1_<tile>_binary.tif → UNet_<tile>.tif
for src_path in sorted(glob.glob(os.path.join(drive_geotiff, 'sentinel-1_*_binary.tif'))):
    fname = os.path.basename(src_path)
    new_name = 'UNet_' + fname.replace('sentinel-1_', '').replace('_binary', '')
    shutil.move(src_path, os.path.join(drive_geotiff, new_name))
    renamed += 1

# Rename probs files: sentinel-1_<tile>_probs.tif → UNet_<tile>_probs.tif
for src_path in sorted(glob.glob(os.path.join(drive_geotiff, 'sentinel-1_*_probs.tif'))):
    fname = os.path.basename(src_path)
    new_name = 'UNet_' + fname.replace('sentinel-1_', '')
    shutil.move(src_path, os.path.join(drive_geotiff, new_name))
    renamed += 1

print(f'✓ Renamed: {renamed}')


> **If rename times out on Drive:** Run the code cell below locally in notebook instead.
> Update `drive_geotiff` to the local path of the `geotiff_predictions` folder.

In [ ]:
import os, glob, shutil

drive_geotiff = r'G:\Min enhet\Examensarbete\Unet_results\goteborg\geotiff_predictions'

renamed = 0
for src_path in sorted(glob.glob(os.path.join(drive_geotiff, '*.tif'))):
    fname = os.path.basename(src_path)
    if fname.startswith('UNet_'):
        continue
    new_name = 'UNet_' + fname.replace('sentinel-1_', '').replace('_binary', '')
    shutil.move(src_path, os.path.join(drive_geotiff, new_name))
    renamed += 1

print(f'✓ Fixed: {renamed}')

## Save results to Google Drive

Save aggregate summary row (first) followed by per-tile metrics to CSV on Drive.

In [ ]:
# Save summary row + per-tile results to Drive CSV
import pandas as pd, os

summary_row = {
    'tile_id': 'SUMMARY',
    'Dataset': 'Sentinel-1',
    'F1': micro_f1,
    'Precision': micro_precision,
    'Recall': micro_recall,
    'IoU': micro_iou,
    'TP': total_TP, 'TN': total_TN, 'FP': total_FP, 'FN': total_FN,
    'accuracy': accuracy,
    'water_f1': water_f1, 'water_precision': water_precision, 'water_recall': water_recall,
    'nw_f1': nw_f1, 'nw_precision': nw_precision, 'nw_recall': nw_recall,
    'macro_f1': macro_f1, 'macro_precision': macro_precision, 'macro_recall': macro_recall,
    'weighted_f1': weighted_f1, 'weighted_precision': weighted_precision, 'weighted_recall': weighted_recall,
}

summary_df = pd.DataFrame([summary_row])
combined_df = pd.concat([summary_df, results_df], ignore_index=True)

csv_path = os.path.join(drive_dirs['tables'], 'unet_results.csv')
combined_df.to_csv(csv_path, index=False)
print(f'✓ Saved to {csv_path}')
print(f'  1 summary row + {len(results_df)} tile rows')